## Pandas for Data Engineering

### 2.1 — Creating & Reading DataFrames

In [1]:
import os
print(os.getcwd())

C:\Users\user\Documents\DataEngineering\Module 1


In [2]:
import pandas as pd

# Create DataFrame
data = [
    {"name": "Gasabo",    "province": "Kigali City", "population": 600000},
    {"name": "Kicukiro",  "province": "Kigali City", "population": 400000},
    {"name": "Musanze",   "province": "Northern",    "population": 420000},
]

df = pd.DataFrame(data)

# Save to CSV
df.to_csv("data.csv", index=False)

# Read it back
df_csv = pd.read_csv("data.csv")

# Explore
print(df_csv.shape)
print(df_csv.dtypes)
print(df_csv.head())
print(df_csv.info())
print(df_csv.describe())

(3, 3)
name          object
province      object
population     int64
dtype: object
       name     province  population
0    Gasabo  Kigali City      600000
1  Kicukiro  Kigali City      400000
2   Musanze     Northern      420000
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   name        3 non-null      object
 1   province    3 non-null      object
 2   population  3 non-null      int64 
dtypes: int64(1), object(2)
memory usage: 204.0+ bytes
None
          population
count       3.000000
mean   473333.333333
std    110151.410946
min    400000.000000
25%    410000.000000
50%    420000.000000
75%    510000.000000
max    600000.000000


### 2.2 Selecting & Filtering Data

In [3]:
# Select one column → returns a Series
print(df["name"])

# Select multiple columns → returns a DataFrame
print(df[["name", "population"]])

# Filter rows by condition
big_districts = df[df["population"] > 400000]

# Filter with multiple conditions — use & and | not 'and'/'or'
kigali_big = df[(df["province"] == "Kigali City") & (df["population"] > 400000)]

# .loc — filter by label/condition (most common)
df.loc[df["population"] > 400000, "name"]

# .iloc — filter by position (row number)
df.iloc[0]      # first row
df.iloc[0:3]    # first 3 rows

0      Gasabo
1    Kicukiro
2     Musanze
Name: name, dtype: object
       name  population
0    Gasabo      600000
1  Kicukiro      400000
2   Musanze      420000


,name,province,population
0,Gasabo,Kigali City,600000
1,Kicukiro,Kigali City,400000
2,Musanze,Northern,420000


### 2.3 — Cleaning Data (the core DE skill)

In [6]:
# --- Check for missing values
print(df.isnull().sum())

# --- Fill missing values
if "population" in df.columns:
    df["population"] = df["population"].fillna(0)

if "province" in df.columns:
    df["province"] = df["province"].fillna("Unknown")

# --- Drop rows with nulls
df = df.dropna()

# --- Remove duplicates
df = df.drop_duplicates(subset=["name"] if "name" in df.columns else None)

# --- Fix data types
if "population" in df.columns:
    df["population"] = df["population"].astype(int)

if "date" in df.columns:
    df["date"] = pd.to_datetime(df["date"])

# --- Clean strings
if "name" in df.columns:
    df["name"] = df["name"].str.strip().str.lower()

if "province" in df.columns:
    df["province"] = df["province"].str.replace("  ", " ")

# --- Rename
df = df.rename(columns={"name": "district_name", "population": "pop"})

# --- Drop safely
df = df.drop(columns=["unnecessary_column"], errors="ignore")

name          0
province      0
population    0
dtype: int64


### 2.4 — Transforming Data

In [8]:
# --- Add a new column
df["pop_millions"] = df["pop"] / 1_000_000

# --- Apply a function
def categorize_size(pop):
    if pop > 500000:
        return "Large"
    elif pop > 300000:
        return "Medium"
    else:
        return "Small"

df["size_category"] = df["pop"].apply(categorize_size)

# --- Group by
summary = df.groupby("province").agg(
    total_population=("pop", "sum"),
    district_count=("district_name", "count"),
    avg_population=("pop", "mean")
).reset_index()

print(summary)

      province  total_population  district_count  avg_population
0  Kigali City           1000000               2        500000.0
1     Northern            420000               1        420000.0


### 2.5 — Merging DataFrames (like SQL JOINs)

In [9]:
# Two tables
districts_df = pd.DataFrame([
    {"district_id": 1, "name": "Gasabo"},
    {"district_id": 2, "name": "Huye"},
])

health_df = pd.DataFrame([
    {"district_id": 1, "health_score": 78},
    {"district_id": 2, "health_score": 85},
])

# INNER JOIN — only matching rows
merged = pd.merge(districts_df, health_df, on="district_id", how="inner")

# LEFT JOIN — all rows from left, match from right
merged = pd.merge(districts_df, health_df, on="district_id", how="left")

### 2.6 — Saving Data

In [10]:
# Save to CSV
df.to_csv("output.csv", index=False)   # index=False — don't save row numbers

# Save to Excel
df.to_excel("output.xlsx", index=False, sheet_name="Districts")

# Save to JSON
df.to_json("output.json", orient="records", indent=2)

## Exercises

In [12]:
import pandas as pd
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s — %(levelname)s — %(message)s"
)

data = [
    {"district": "Gasabo",     "province": "Kigali City", "population": 600000, "health_score": 72, "poverty_rate": 15.2},
    {"district": "Kicukiro",   "province": "Kigali City", "population": 400000, "health_score": 68, "poverty_rate": 18.5},
    {"district": "Nyarugenge", "province": "Kigali City", "population": 350000, "health_score": None,"poverty_rate": 20.1},
    {"district": "Huye",       "province": "Southern",    "population": 330000, "health_score": 61, "poverty_rate": 35.0},
    {"district": "Musanze",    "province": "Northern",    "population": 420000, "health_score": 65, "poverty_rate": 28.3},
    {"district": "Rubavu",     "province": "Western",     "population": 390000, "health_score": 63, "poverty_rate": 30.1},
    {"district": "Gasabo",     "province": "Kigali City", "population": 600000, "health_score": 72, "poverty_rate": 15.2},
]

df = pd.DataFrame(data)

Exercise 1 — Load the data into a DataFrame. Print its shape, data types, and count of null values per column.

In [13]:
def inspect_dataframe(df: pd.DataFrame) -> None:
    logging.info(f"Shape: {df.shape}")
    print("\nData Types:\n", df.dtypes)
    print("\nNull Values:\n", df.isnull().sum())
    print("\nFirst Rows:\n", df.head())

inspect_dataframe(df)

2026-04-17 15:49:07,721 — INFO — Shape: (7, 5)



Data Types:
 district         object
province         object
population        int64
health_score    float64
poverty_rate    float64
dtype: object

Null Values:
 district        0
province        0
population      0
health_score    1
poverty_rate    0
dtype: int64

First Rows:
      district     province  population  health_score  poverty_rate
0      Gasabo  Kigali City      600000          72.0          15.2
1    Kicukiro  Kigali City      400000          68.0          18.5
2  Nyarugenge  Kigali City      350000           NaN          20.1
3        Huye     Southern      330000          61.0          35.0
4     Musanze     Northern      420000          65.0          28.3


Exercise 2 — Clean the data: fill the missing `health_score` with the column average, remove the duplicate row, and rename `poverty_rate` to `poverty_pct`.

In [14]:
def clean_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Fill missing health_score with mean
    mean_health = df["health_score"].mean()
    df["health_score"] = df["health_score"].fillna(mean_health)

    # Remove duplicates
    df = df.drop_duplicates()

    # Rename column
    df = df.rename(columns={"poverty_rate": "poverty_pct"})

    return df


df_clean = clean_dataframe(df)

print(df_clean)

     district     province  population  health_score  poverty_pct
0      Gasabo  Kigali City      600000     72.000000         15.2
1    Kicukiro  Kigali City      400000     68.000000         18.5
2  Nyarugenge  Kigali City      350000     66.833333         20.1
3        Huye     Southern      330000     61.000000         35.0
4     Musanze     Northern      420000     65.000000         28.3
5      Rubavu      Western      390000     63.000000         30.1


✔ Key DE ideas:

-`.copy()` avoids modifying original data
- Mean imputation is common for numeric columns
- Deduplication before analysis

Exercise 3 — Add a new column `health_category:` `"Good"` if health_score ≥ 70, `"Fair"` if ≥ 60, `"Poor"` if below 60. Use `.apply()`.

In [18]:
def categorize_health(score: float) -> str:
    if score >= 70:
        return "Good"
    elif score >= 60:
        return "Fair"
    else:
        return "Poor"


df_clean["health_category"] = df_clean["health_score"].apply(categorize_health)

print(df_clean[["district", "health_score", "health_category"]])

     district  health_score health_category
0      Gasabo     72.000000            Good
1    Kicukiro     68.000000            Fair
2  Nyarugenge     66.833333            Fair
3        Huye     61.000000            Fair
4     Musanze     65.000000            Fair
5      Rubavu     63.000000            Fair


Exercise 4 — Group by `province` and calculate total population, average health score, and average poverty rate per province. Sort results by total population descending.

In [19]:
def summarize_by_province(df: pd.DataFrame) -> pd.DataFrame:
    summary = (
        df.groupby("province")
        .agg(
            total_population=("population", "sum"),
            avg_health_score=("health_score", "mean"),
            avg_poverty_pct=("poverty_pct", "mean")
        )
        .reset_index()
        .sort_values(by="total_population", ascending=False)
    )

    return summary


summary_df = summarize_by_province(df_clean)

print(summary_df)

      province  total_population  avg_health_score  avg_poverty_pct
0  Kigali City           1350000         68.944444        17.933333
1     Northern            420000         65.000000        28.300000
3      Western            390000         63.000000        30.100000
2     Southern            330000         61.000000        35.000000


Exercise 5 — Save the cleaned, transformed DataFrame to both a CSV and an Excel file.

In [20]:
def save_outputs(df: pd.DataFrame, base_filename: str = "districts_cleaned"):
    csv_file = f"{base_filename}.csv"
    excel_file = f"{base_filename}.xlsx"

    df.to_csv(csv_file, index=False)
    df.to_excel(excel_file, index=False, sheet_name="Cleaned Data")

    logging.info(f"Saved CSV: {csv_file}")
    logging.info(f"Saved Excel: {excel_file}")


save_outputs(df_clean)

2026-04-17 16:08:54,885 — INFO — Saved CSV: districts_cleaned.csv
2026-04-17 16:08:54,886 — INFO — Saved Excel: districts_cleaned.xlsx


## Full Pipeline (How a Data Engineer Would Combine It)

This is the part most people miss — stitching everything together: 

In [21]:
def run_pipeline():
    logging.info("Starting pipeline...")

    df = pd.DataFrame(data)

    inspect_dataframe(df)

    df_clean = clean_dataframe(df)

    df_clean["health_category"] = df_clean["health_score"].apply(categorize_health)

    summary = summarize_by_province(df_clean)

    save_outputs(df_clean)

    logging.info("Pipeline completed successfully")

    return df_clean, summary


df_final, summary_final = run_pipeline()

2026-04-17 16:10:15,828 — INFO — Starting pipeline...
2026-04-17 16:10:15,830 — INFO — Shape: (7, 5)
2026-04-17 16:10:15,901 — INFO — Saved CSV: districts_cleaned.csv
2026-04-17 16:10:15,903 — INFO — Saved Excel: districts_cleaned.xlsx
2026-04-17 16:10:15,905 — INFO — Pipeline completed successfully



Data Types:
 district         object
province         object
population        int64
health_score    float64
poverty_rate    float64
dtype: object

Null Values:
 district        0
province        0
population      0
health_score    1
poverty_rate    0
dtype: int64

First Rows:
      district     province  population  health_score  poverty_rate
0      Gasabo  Kigali City      600000          72.0          15.2
1    Kicukiro  Kigali City      400000          68.0          18.5
2  Nyarugenge  Kigali City      350000           NaN          20.1
3        Huye     Southern      330000          61.0          35.0
4     Musanze     Northern      420000          65.0          28.3
